# Customer Churn & E-commerce Sales Trend Analysis

**Project objective:** Analyze real e-commerce transaction data to understand revenue trends, identify valuable customer segments, find top-selling products and countries, and estimate customer inactivity using an RFM-style analysis.

**Dataset:** UCI Online Retail dataset. It contains invoice dates, products, quantities, prices, customer IDs, and countries.

> This is a transaction dataset, not a subscription dataset. Therefore, churn below is an *inactivity-based proxy* (customers with no purchase in the last 90 days of the observed data), not confirmed business cancellation.

Run each cell from top to bottom.

## 1. Import libraries

In [ ]:
import io
import zipfile
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 50)

print("Libraries imported successfully!")

## 2. Download and load the real dataset

In [ ]:
# UCI Online Retail dataset (Excel file inside a ZIP archive)
url = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"

response = requests.get(url, timeout=60)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
    excel_file = next(name for name in archive.namelist()
                      if name.lower().endswith((".xlsx", ".xls")))
    with archive.open(excel_file) as file:
        raw = pd.read_excel(file)

df = raw.copy()

print("Dataset loaded:", df.shape)
display(df.head())

## 3. Understand the dataset

In [ ]:
print("Dataset dimensions:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()

print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing values"))

print("\nNumerical summary:")
display(df.describe().T)

## 4. Clean and prepare transactions

In [ ]:
# Standardize column names
df.columns = df.columns.str.strip()

# Convert date column to datetime
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

# Remove rows missing essential transaction fields
df = df.dropna(subset=["InvoiceNo", "StockCode", "Description",
                       "InvoiceDate", "Quantity", "UnitPrice"])

# Convert types
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")

# Remove invalid quantity/price records
df = df.dropna(subset=["Quantity", "UnitPrice"])
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]

# Remove cancelled invoices (invoice numbers beginning with C)
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df = df[~df["InvoiceNo"].str.upper().str.startswith("C")]

# Revenue per line item
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

# Customer ID is needed for customer-level analysis
df["CustomerID"] = df["CustomerID"].astype("Int64")

print("Cleaned transaction rows:", len(df))
print("Unique invoices:", df["InvoiceNo"].nunique())
print("Unique customers with IDs:", df["CustomerID"].nunique())
print("Date range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())
display(df.head())

## 5. Overall sales KPIs

In [ ]:
total_revenue = df["Revenue"].sum()
total_orders = df["InvoiceNo"].nunique()
total_customers = df["CustomerID"].nunique()
total_units = df["Quantity"].sum()
average_order_value = total_revenue / total_orders if total_orders else 0

kpis = pd.DataFrame({
    "Metric": ["Total revenue", "Unique orders", "Customers with ID",
               "Units sold", "Average order value"],
    "Value": [total_revenue, total_orders, total_customers,
              total_units, average_order_value]
})
display(kpis.style.format({"Value": lambda x: f"{x:,.2f}"}))

## 6. Monthly revenue trend

In [ ]:
monthly_sales = (
    df.set_index("InvoiceDate")
      .resample("MS")["Revenue"]
      .sum()
      .reset_index()
)

plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly_sales, x="InvoiceDate", y="Revenue", marker="o")
plt.title("Monthly E-commerce Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

display(monthly_sales.tail(12))

## 7. Monthly order volume

In [ ]:
monthly_orders = (
    df.set_index("InvoiceDate")
      .resample("MS")["InvoiceNo"]
      .nunique()
      .reset_index(name="Orders")
)

plt.figure(figsize=(12, 5))
sns.barplot(data=monthly_orders, x="InvoiceDate", y="Orders")
plt.title("Number of Orders per Month")
plt.xlabel("Month")
plt.ylabel("Unique orders")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Top 10 countries by revenue

In [ ]:
country_sales = (
    df.groupby("Country", as_index=False)
      .agg(Revenue=("Revenue", "sum"),
           Orders=("InvoiceNo", "nunique"),
           Customers=("CustomerID", "nunique"))
      .sort_values("Revenue", ascending=False)
)

top_countries = country_sales.head(10)

plt.figure(figsize=(11, 6))
sns.barplot(data=top_countries, y="Country", x="Revenue")
plt.title("Top 10 Countries by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

display(top_countries)

## 9. Top 10 products by revenue

In [ ]:
product_sales = (
    df.groupby(["StockCode", "Description"], as_index=False)
      .agg(Revenue=("Revenue", "sum"),
           Units_Sold=("Quantity", "sum"))
      .sort_values("Revenue", ascending=False)
)

top_products = product_sales.head(10)

plt.figure(figsize=(12, 6))
sns.barplot(data=top_products, y="Description", x="Revenue")
plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

display(top_products)

## 10. Customer revenue distribution

In [ ]:
customer_sales = (
    df.dropna(subset=["CustomerID"])
      .groupby("CustomerID", as_index=False)
      .agg(Revenue=("Revenue", "sum"),
           Orders=("InvoiceNo", "nunique"),
           Units=("Quantity", "sum"),
           LastPurchase=("InvoiceDate", "max"))
      .sort_values("Revenue", ascending=False)
)

plt.figure(figsize=(9, 5))
sns.histplot(customer_sales["Revenue"], bins=50, kde=True)
plt.title("Distribution of Customer Lifetime Revenue")
plt.xlabel("Customer revenue")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()

display(customer_sales.head(10))

## 11. Repeat customer analysis

In [ ]:
orders_per_customer = (
    df.dropna(subset=["CustomerID"])
      .groupby("CustomerID")["InvoiceNo"]
      .nunique()
)

repeat_summary = pd.Series({
    "One-time customers": int((orders_per_customer == 1).sum()),
    "Repeat customers (2+ orders)": int((orders_per_customer >= 2).sum())
})

display(repeat_summary.to_frame("Customer count"))

plt.figure(figsize=(7, 5))
sns.barplot(x=repeat_summary.index, y=repeat_summary.values)
plt.title("One-time vs Repeat Customers")
plt.xlabel("Customer type")
plt.ylabel("Number of customers")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## 12. RFM customer segmentation

In [ ]:
# RFM = Recency, Frequency, Monetary value
# Use the day after the final observed transaction as the analysis date.
customer_df = df.dropna(subset=["CustomerID"]).copy()
analysis_date = customer_df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = (
    customer_df.groupby("CustomerID")
    .agg(
        Recency=("InvoiceDate", lambda x: (analysis_date - x.max()).days),
        Frequency=("InvoiceNo", "nunique"),
        Monetary=("Revenue", "sum")
    )
)

# Score each customer from 1 (low) to 4 (high).
# Lower recency is better, so reverse the Recency score.
rfm["R_Score"] = pd.qcut(
    rfm["Recency"].rank(method="first"),
    4, labels=[4, 3, 2, 1]
).astype(int)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    4, labels=[1, 2, 3, 4]
).astype(int)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    4, labels=[1, 2, 3, 4]
).astype(int)

rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str) +
    rfm["F_Score"].astype(str) +
    rfm["M_Score"].astype(str)
)

def segment_customer(row):
    if row["R_Score"] >= 3 and row["F_Score"] >= 3 and row["M_Score"] >= 3:
        return "Best customers"
    elif row["R_Score"] >= 3 and row["F_Score"] >= 2:
        return "Loyal / active"
    elif row["R_Score"] <= 2 and row["F_Score"] >= 3:
        return "At risk"
    elif row["R_Score"] == 1:
        return "Inactive"
    else:
        return "Potential / occasional"

rfm["Segment"] = rfm.apply(segment_customer, axis=1)

segment_summary = (
    rfm.groupby("Segment")
       .agg(Customers=("Segment", "size"),
            Average_Revenue=("Monetary", "mean"),
            Total_Revenue=("Monetary", "sum"))
       .sort_values("Customers", ascending=False)
)

display(segment_summary.round(2))

## 13. Visualize customer segments

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=rfm, y="Segment",
              order=rfm["Segment"].value_counts().index)
plt.title("Number of Customers by RFM Segment")
plt.xlabel("Number of customers")
plt.ylabel("Customer segment")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
segment_revenue = rfm.groupby("Segment")["Monetary"].sum().sort_values(ascending=False)
sns.barplot(x=segment_revenue.values, y=segment_revenue.index)
plt.title("Total Revenue by Customer Segment")
plt.xlabel("Revenue")
plt.ylabel("Customer segment")
plt.tight_layout()
plt.show()

## 14. Inactivity-based churn proxy

In [ ]:
# This dataset has no explicit churn/cancellation label.
# Define an inactivity proxy: no purchase in the last 90 days
# of the observed transaction period.
last_observed_date = customer_df["InvoiceDate"].max()
customer_last_purchase = customer_df.groupby("CustomerID")["InvoiceDate"].max()
days_inactive = (last_observed_date - customer_last_purchase).dt.days

churn_proxy = pd.DataFrame({
    "LastPurchase": customer_last_purchase,
    "DaysInactive": days_inactive
})
churn_proxy["InactivityStatus"] = np.where(
    churn_proxy["DaysInactive"] > 90,
    "Inactive (90+ days)",
    "Active (last 90 days)"
)

status_counts = churn_proxy["InactivityStatus"].value_counts()
display(status_counts.to_frame("Customers"))

plt.figure(figsize=(7, 5))
sns.barplot(x=status_counts.index, y=status_counts.values)
plt.title("Customer Inactivity Proxy (90-day threshold)")
plt.xlabel("Status")
plt.ylabel("Number of customers")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

print("Note: This is a recency-based inactivity proxy, not confirmed churn.")

## 15. Revenue correlation matrix and heatmap

In [ ]:
# Aggregate numerical customer-level metrics for correlation
customer_metrics = (
    customer_df.groupby("CustomerID")
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("InvoiceNo", "nunique"),
        Quantity=("Quantity", "sum")
    )
)

customer_metrics["AverageOrderValue"] = (
    customer_metrics["Revenue"] / customer_metrics["Orders"]
)

corr = customer_metrics.corr(numeric_only=True)
display(corr.round(2))

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm",
            center=0, fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix of Customer Metrics")
plt.tight_layout()
plt.show()

## 16. Final business insights

In [ ]:
inactive_count = int((days_inactive > 90).sum())
customer_count = int(customer_last_purchase.shape[0])
inactive_pct = inactive_count / customer_count * 100 if customer_count else 0

print("========== E-COMMERCE ANALYSIS REPORT ==========")
print(f"Analysis period: {df['InvoiceDate'].min().date()} to {df['InvoiceDate'].max().date()}")
print(f"Total revenue: {total_revenue:,.2f}")
print(f"Unique orders: {total_orders:,}")
print(f"Customers with IDs: {total_customers:,}")
print(f"Average order value: {average_order_value:,.2f}")
print(f"Top revenue country: {top_countries.iloc[0]['Country']}")
print(f"Top revenue product: {top_products.iloc[0]['Description']}")
print(f"Customers inactive for over 90 days: {inactive_count:,} ({inactive_pct:.2f}%)")
print("\nCustomer segments:")
print(segment_summary[["Customers", "Total_Revenue"]].round(2))
print("================================================")

## 17. Export analysis outputs

In [ ]:
# Save cleaned transaction data and customer analysis tables
df.to_csv("cleaned_ecommerce_transactions.csv", index=False)
rfm.to_csv("customer_rfm_segments.csv")
monthly_sales.to_csv("monthly_revenue_trend.csv", index=False)
country_sales.to_csv("country_sales_summary.csv", index=False)
product_sales.to_csv("product_sales_summary.csv", index=False)

print("CSV reports saved in the Colab working directory.")

# Optional: download the RFM customer segmentation file
from google.colab import files
files.download("customer_rfm_segments.csv")

## Conclusion

This project analyzed real e-commerce transactions to understand sales performance and customer behavior. It measured revenue and order trends over time, identified top countries and products, compared one-time and repeat buyers, and segmented customers using Recency, Frequency, and Monetary value (RFM).

The 90-day inactivity measure is only a churn proxy because the dataset does not contain an explicit churn label. Business recommendations should be based on the actual values and charts generated in the notebook.